# Tiny Transformer

This notebook builds the first GPT-style Transformer pass on top of the tokenizer and batching work.

The model receives `x` token tensors shaped `(batch_size, context_length)` and predicts the next token for every position. The target tensor `y` has the same shape, shifted one token forward.


## Colab Bootstrap

Run this cell first. If you already ran `colab_setup.ipynb` in the same Colab runtime, this will reuse that setup; if not, it installs the small dependency set and clones the repo.


In [19]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/markschatza/llm-from-scratch.git"
REPO_REF = "codex/colab-setup-workflow"
REPO_DIR = Path("/content/llm-from-scratch")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pretrain" / "tokenizer.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find the repo root. Run this notebook from inside the llm-from-scratch checkout."
    )

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "numpy", "torch"], check=True)
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF], check=True)
    project_root = REPO_DIR
else:
    project_root = find_project_root(Path.cwd())

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"project root: {project_root}")
print(f"repo ref: {REPO_REF if IN_COLAB else 'local checkout'}")
print(f"in colab: {IN_COLAB}")


project root: C:\git\LLMFromScratch
repo ref: local checkout
in colab: False


## Imports


In [20]:
from pathlib import Path
import time

import torch

from pretrain.batching import BatchConfig, get_batch, load_token_ids, split_tokens
from pretrain.tokenizer import Tokenizer
from pretrain.transformer import GPTLanguageModel, TransformerConfig


## Prepare Token IDs

If the tokenizer output does not exist yet, create it from the small starter corpus.


In [21]:
CORPUS = Path("data/van-life-story.txt")
OUTPUT_DIR = Path("pretrain/data")
TOKEN_BIN = OUTPUT_DIR / "train.van-life-story.bin"
MODEL_PATH = OUTPUT_DIR / "sp.model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not TOKEN_BIN.exists() or not MODEL_PATH.exists():
    tok = Tokenizer.train(CORPUS, vocab_size=1024, output_dir=OUTPUT_DIR, name="sp")
    all_tokens: list[int] = []
    with open(CORPUS, "r", encoding="utf-8") as fin:
        for line in fin:
            line = line.rstrip("\n")
            if line:
                all_tokens.extend(tok.encode(line))
    torch.tensor(all_tokens, dtype=torch.int64).numpy().astype("uint32").tofile(TOKEN_BIN)

print(f"token file: {TOKEN_BIN}")
print(f"tokenizer model: {MODEL_PATH}")


token file: pretrain\data\train.van-life-story.bin
tokenizer model: pretrain\data\sp.model


## Build A Batch


In [22]:
def pick_training_device() -> str:
    """Prefer the GPU backend when PyTorch can see one, otherwise use CPU."""
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


device = pick_training_device()
print(f"torch: {torch.__version__}")
print(f"device: {device}")
if device == "cuda":
    print(f"gpu: {torch.cuda.get_device_name(0)}")
    print(f"hip: {getattr(torch.version, 'hip', None)}")
else:
    print("gpu: not available; training will run on CPU")

tokens = load_token_ids(TOKEN_BIN)
train_tokens, val_tokens = split_tokens(tokens, train_fraction=0.9)

# For GPU training, keep the token corpus on the GPU so each batch is sampled there.
train_tokens_for_batches = train_tokens.to(device)
val_tokens_for_batches = val_tokens.to(device)

batch_config = BatchConfig(
    batch_size=64 if device == "cuda" else 4,
    context_length=128 if device == "cuda" else 16,
    device=device,
)
generator = torch.Generator().manual_seed(123)
x, y = get_batch(train_tokens_for_batches, batch_config, generator=generator)

print(f"train tokens: {len(train_tokens):,}")
print(f"batch size: {batch_config.batch_size}")
print(f"context length: {batch_config.context_length}")
print(f"x shape: {tuple(x.shape)} on {x.device}")
print(f"y shape: {tuple(y.shape)} on {y.device}")
print(f"shift check: {torch.equal(x[:, 1:], y[:, :-1])}")


torch: 2.9.1+rocm7.2.1
device: cuda
gpu: AMD Radeon RX 7900 GRE
hip: 7.2.53211-158bd99533
train tokens: 12,057
batch size: 64
context length: 128
x shape: (64, 128) on cuda:0
y shape: (64, 128) on cuda:0
shift check: True


## Instantiate The Transformer

This is intentionally tiny: enough to prove the mechanics before we scale dimensions, layers, context length, or training time.


In [23]:
tok = Tokenizer.from_pretrained(MODEL_PATH)

model_config = TransformerConfig(
    vocab_size=tok.vocab_size,
    context_length=batch_config.context_length,
    n_embd=256 if device == "cuda" else 64,
    n_head=8 if device == "cuda" else 4,
    n_layer=4 if device == "cuda" else 2,
    dropout=0.0,
)
model = GPTLanguageModel(model_config).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model_config)
print(f"parameters: {n_params:,}")
print(f"model device: {next(model.parameters()).device}")


TransformerConfig(vocab_size=1024, context_length=128, n_embd=256, n_head=8, n_layer=4, dropout=0.0)
parameters: 3,717,632
model device: cuda:0


## Forward Pass And Loss


In [24]:
logits, loss = model(x, y)

print(f"model device: {next(model.parameters()).device}")
print(f"batch device: {x.device}")
print(f"logits shape: {tuple(logits.shape)} on {logits.device}")
print(f"loss: {loss.item():.4f}")
print(f"expected logits shape: {(batch_config.batch_size, batch_config.context_length, tok.vocab_size)}")


model device: cuda:0
batch device: cuda:0
logits shape: (64, 128, 1024) on cuda:0
loss: 6.9772
expected logits shape: (64, 128, 1024)


## Train On The Selected Device

On GPU, this uses a larger toy model and larger batches than the CPU path. The token corpus is kept on the selected device, and batches are sampled with vectorized tensor indexing so the loop spends less time in Python and more time doing Transformer math.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
train_generator = torch.Generator().manual_seed(456)
train_steps = 1000000 if device == "cuda" else 100
log_every = 1000 if device == "cuda" else 10

tokens_per_step = batch_config.batch_size * batch_config.context_length
print(f"training on: {next(model.parameters()).device}")
print(f"steps: {train_steps}")
print(f"tokens per step: {tokens_per_step:,}")

if device == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()

model.train()
for step in range(train_steps + 1):
    xb, yb = get_batch(train_tokens_for_batches, batch_config, generator=train_generator)
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % log_every == 0 or step == train_steps:
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = max(time.perf_counter() - t0, 1e-9)
        tokens_per_second = ((step + 1) * tokens_per_step) / elapsed
        print(
            f"step {step:04d} | loss {loss.item():.4f} | "
            f"{tokens_per_second:,.0f} tok/s | batch {xb.device} | model {next(model.parameters()).device}"
        )


training on: cuda:0
steps: 1000
tokens per step: 8,192
step 0000 | loss 6.9833 | 15,421 tok/s | batch cuda:0 | model cuda:0
step 0050 | loss 5.7246 | 173,155 tok/s | batch cuda:0 | model cuda:0
step 0100 | loss 4.9279 | 192,422 tok/s | batch cuda:0 | model cuda:0
step 0150 | loss 3.5816 | 199,943 tok/s | batch cuda:0 | model cuda:0
step 0200 | loss 1.5550 | 203,896 tok/s | batch cuda:0 | model cuda:0
step 0250 | loss 0.3004 | 206,301 tok/s | batch cuda:0 | model cuda:0
step 0300 | loss 0.1548 | 207,981 tok/s | batch cuda:0 | model cuda:0
step 0350 | loss 0.1200 | 209,186 tok/s | batch cuda:0 | model cuda:0
step 0400 | loss 0.0879 | 210,079 tok/s | batch cuda:0 | model cuda:0
step 0450 | loss 0.0819 | 210,774 tok/s | batch cuda:0 | model cuda:0
step 0500 | loss 0.0847 | 211,357 tok/s | batch cuda:0 | model cuda:0
step 0550 | loss 0.0743 | 211,774 tok/s | batch cuda:0 | model cuda:0
step 0600 | loss 0.0703 | 212,146 tok/s | batch cuda:0 | model cuda:0
step 0650 | loss 0.0584 | 212,455 to

## Generate A Tiny Sample


In [27]:
model.eval()
tok = Tokenizer.from_pretrained(MODEL_PATH)
start = torch.tensor(tok.encode("Once upon a time"), dtype=torch.long, device=device).unsqueeze(0)
out = model.generate(start, max_new_tokens=40)
ids = out[0].tolist()

print(ids)
print()
print(tok.decode(ids))


[988, 787, 482, 290, 258, 594, 299, 590, 533, 299, 921, 971, 965, 728, 962, 369, 320, 976, 367, 351, 310, 285, 306, 967, 908, 258, 263, 335, 539, 339, 970, 719, 445, 972, 611, 345, 287, 960, 291, 258, 507, 328, 279, 792, 451, 283]

Once upon a time to anyact to explainycialriece, because he had plyw truck a s is approp entire professionals in a team culture was
